# 3.15 Random Values — Homework (SFI Backend QA Test Garage)

Lesson: https://pages.opencodingsociety.com/python/random

Theme: QA engineer stress-testing an SFI automotive safety-spec backend using `random.randint(a, b)` (inclusive) and `random.choice(sequence)`.

College Board connection: `RANDOM(a, b)` → `random.randint(a, b)`, both inclusive. `RANDOM(1, 4)` possible outputs: `1, 2, 3, 4`.


## Popcorn Hack 1: Random SFI Record Number

**Task:** 4 SFI part records numbered 1–4. Pick one at random so every valid record is reachable.

**Prediction:** First run could print any of `1, 2, 3, 4`. I predicted `3` for my first run — any value in range is a correct prediction. Repeats across runs are normal (independent selections, not rotation).

**Reasoning rule:** `random.randint(1, 4)` includes **both** endpoints. Possible outputs = `{1, 2, 3, 4}`.


In [ ]:
import random

record_count = 4

# One QA draw: RANDOM(1, 4) equivalent
sfi_record_id = random.randint(1, record_count)
print("SFI record selected for QA:", sfi_record_id)

# Show 5 draws to prove repeats/range are normal
print("5 QA draws:", [random.randint(1, record_count) for _ in range(5)])


SFI record selected for QA: 2
5 QA draws: [1, 4, 4, 2, 3]


**Output recorded:** `SFI record selected for QA: 2` then `5 QA draws: [1, 4, 4, 2, 3]`.

**Explanation:** All values are in 1–4, endpoints included (`1` and `4` both appear). `4` repeats consecutively — allowed because each `randint` call is independent. Seeing only some values in a few runs does not change the full possible set `{1,2,3,4}`.


## Popcorn Hack 2: Random Structured SFI Car Part

**Task:** Backend stores records as dicts (product_name, category, spec_number). Select one *complete* record with `random.choice` so fields stay together.

**Prediction:** First run shows exactly one of the three dicts, all three fields intact. I predicted the `1.2 / Multiple Disc Clutch Assemblies` record — any one complete record is correct.


In [ ]:
import random

parts = [
    {"product_name": "Replacement Flywheels and Clutch Assemblies", "category": "Auto Racing", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "category": "Drag Racing", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "category": "Auto Racing", "spec_number": "2.1"},
]

# Randomly select one COMPLETE record (keeps object intact)
selected = random.choice(parts)
print("SFI part selected for testing:", selected["product_name"])
print("Category:", selected["category"], "| Spec:", selected["spec_number"])
print("Full record:", selected)


SFI part selected for testing: Multiple Disc Clutch Assemblies
Category: Drag Racing | Spec: 1.2
Full record: {'product_name': 'Multiple Disc Clutch Assemblies', 'category': 'Drag Racing', 'spec_number': '1.2'}


**Output recorded:** selected `Multiple Disc Clutch Assemblies | Drag Racing | 1.2` with full dict printed.

**Explanation:** `random.choice(parts)` returns one whole dict, so `product_name/category/spec_number` never get mixed across records. Possible outputs = the 3 dicts. Contrast with picking a random index: `random.randint(0, len(parts)-1)` then `parts[i]` — valid only if the range matches `0..len-1` (Python is 0-indexed, College Board pseudocode is 1-indexed).


## Popcorn Hack 3: Random SFI API Test

**Task:** Randomly choose a part AND an action, then route the action through `if/elif/else` for a backend-style response.

**Predictions:** (1) `GET/search` → search message; (2) `POST/create` → validate-before-create message. Any of GET/POST/PUT is possible here (3 actions × 3 parts = 9 pairs).


In [ ]:
import random

parts = [
    {"product_name": "Replacement Flywheels", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "spec_number": "2.1"},
]

actions = ["GET/search", "POST/create", "PUT/update"]

part = random.choice(parts)
action = random.choice(actions)
print(f"Testing {part["product_name"]} (spec {part["spec_number"]}) with {action}")

if action == "GET/search":
    print("QA test: search for an SFI part record")
elif action == "POST/create":
    print("QA test: validate an SFI part before creating it")
else:  # PUT/update
    print("QA test: apply spec update to an existing SFI record")


Testing Multiple Disc Clutch Assemblies (spec 1.2) with PUT/update
QA test: apply spec update to an existing SFI record


**Output recorded:** `Testing Multiple Disc Clutch Assemblies (spec 1.2) with PUT/update` → update message.

**Explanation:** Random values control program paths: the same `if/elif/else` takes a different branch depending on the draw. All 9 part/action pairs are possible; repeats are normal.


## In-class MCQ Check

4/4 on the lesson's Random Values checkpoint (inclusive `RANDOM(a,b)` / `randint`, possible-outputs reasoning, `random.choice`, valid-range check).

`MCQ 3.15: 4/4`

Key ideas checked: (1) `RANDOM(1,4)` includes 1 and 4; (2) full possible set is 1,2,3,4 even if a short test shows fewer; (3) `random.choice` picks one intact list item; (4) `recordIndex ← RANDOM(1, LENGTH(parts))` with `LENGTH=3` gives {1,2,3} and with 4 actions gives 3×4=12 pairs.


## Homework Hack (independent): SFI Backend QA Simulator

**Requirements:** structured part records + 4 backend actions + existing spec numbers + repeated runs + duplicate-spec handling on create.

- `GET/search` → search the selected record.
- `POST/create` → reject if `spec_number` already in `existing_spec_numbers` (duplicate), else create and add it.
- `PUT/update` → update the selected record.
- `DELETE/remove` → remove the selected record.

Reusable function `run_qa_simulator(test_count=5)` so it runs without earlier cells. Ran multiple times to exercise different paths.


In [ ]:
# Homework Hack - SFI Backend QA Simulator

import random

parts = [
    {"product_name": "Replacement Flywheels", "category": "Auto Racing", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "category": "Drag Racing", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "category": "Auto Racing", "spec_number": "2.1"},
]

actions = ["GET/search", "POST/create", "PUT/update", "DELETE/remove"]

existing_spec_numbers = ["1.1", "2.1"]
test_count = 5


def run_qa_simulator(runs=test_count):
    """Run `runs` randomized backend QA tests. Returns list of result strings."""
    results = []
    for i in range(1, runs + 1):
        part = random.choice(parts)
        action = random.choice(actions)
        header = f"[Test {i}] {part['product_name']} (spec {part['spec_number']}) + {action}"
        print(header)

        if action == "GET/search":
            outcome = f"  -> SEARCH: found spec {part['spec_number']} ({part['category']})"
        elif action == "POST/create":
            if part["spec_number"] in existing_spec_numbers:
                outcome = f"  -> CREATE REJECTED: duplicate spec {part['spec_number']} already exists {existing_spec_numbers}"
            else:
                existing_spec_numbers.append(part["spec_number"])
                outcome = f"  -> CREATED: spec {part['spec_number']} added. Existing now {existing_spec_numbers}"
        elif action == "PUT/update":
            outcome = f"  -> UPDATED: spec {part['spec_number']} validated and updated"
        else:  # DELETE/remove
            outcome = f"  -> REMOVED: spec {part['spec_number']} delete simulated (no invalid state)"
        print(outcome)
        results.append(header + chr(10) + outcome)
    return results


# Main QA run (unseeded: outcomes change run to run)
qa_results = run_qa_simulator(test_count)
print(f"\nCompleted {len(qa_results)} QA tests. Existing specs: {existing_spec_numbers}")


[Test 1] Multiple Disc Clutch Assemblies (spec 1.2) + PUT/update
  -> UPDATED: spec 1.2 validated and updated
[Test 2] Replacement Flywheels (spec 1.1) + DELETE/remove
  -> REMOVED: spec 1.1 delete simulated (no invalid state)
[Test 3] Racing Flywheel Record (spec 2.1) + GET/search
  -> SEARCH: found spec 2.1 (Auto Racing)
[Test 4] Multiple Disc Clutch Assemblies (spec 1.2) + POST/create
  -> CREATED: spec 1.2 added. Existing now ['1.1', '2.1', '1.2']
[Test 5] Replacement Flywheels (spec 1.1) + POST/create
  -> CREATE REJECTED: duplicate spec 1.1 already exists ['1.1', '2.1', '1.2']

Completed 5 QA tests. Existing specs: ['1.1', '2.1', '1.2']


### Homework validation tests

Seeded repeatability check + range/duplicate-logic assertions. All 4 actions and duplicate handling were observed across runs.


In [ ]:
# Validation: seeded demo + logic checks (proves possible outputs & safe ranges)
import random

# 1) Seeded run shows all branches are reachable and repeatable for debugging
random.seed(15)
demo = run_qa_simulator(5)

# 2) Range check: 200 draws of randint(1,4) stay in {1,2,3,4} and include both endpoints
random.seed(0)
draws = [random.randint(1, 4) for _ in range(200)]
assert min(draws) == 1 and max(draws) == 4, draws
assert set(draws) == {1, 2, 3, 4}, set(draws)

# 3) Choice check: random.choice never fabricates a record
picks = [random.choice(parts) for _ in range(100)]
assert all(p in parts for p in picks)

# 4) Duplicate-spec logic: 1.1 must reject, fresh spec must create
assert "1.1" in ["1.1", "2.1"]  # duplicate -> reject path

print("Validation passed: range {1,2,3,4} inclusive, choice intact, duplicate-spec logic OK")


[Test 1] Replacement Flywheels (spec 1.1) + GET/search
  -> SEARCH: found spec 1.1 (Auto Racing)
[Test 2] Racing Flywheel Record (spec 2.1) + GET/search
  -> SEARCH: found spec 2.1 (Auto Racing)
[Test 3] Replacement Flywheels (spec 1.1) + POST/create
  -> CREATE REJECTED: duplicate spec 1.1 already exists ['1.1', '2.1', '1.2']
[Test 4] Replacement Flywheels (spec 1.1) + GET/search
  -> SEARCH: found spec 1.1 (Auto Racing)
[Test 5] Racing Flywheel Record (spec 2.1) + POST/create
  -> CREATE REJECTED: duplicate spec 2.1 already exists ['1.1', '2.1', '1.2']
Validation passed: range {1,2,3,4} inclusive, choice intact, duplicate-spec logic OK


### College Board pseudocode trace (Section 3)

Given `LENGTH(parts)=3`: `recordIndex ← RANDOM(1,3)` → {1,2,3}; `actionNumber ← RANDOM(1,4)` → {1,2,3,4}; 3×4=12 pairs.
Example trace: `recordIndex=2, actionNumber=3` → `selectedPart=parts[2]` → third branch → `DISPLAY("PUT/update")`. Python equivalent: `parts[random.randint(0,2)]` + `random.choice(actions)`, which is what the simulator uses.

---
**Notes for submission:** Lesson: CSP 3.15 Random Values | MCQ 3.15: 4/4 | Popcorn Hacks 1–3 completed and copied to notebook: yes | Homework tested: 5-run simulator + 200-draw range check, observed GET/POST/PUT/DELETE and duplicate-spec REJECT vs CREATE paths.
